Nom : Khadijetou Mohamed abe

Filière : GLCC

## fonctions communes

In [1]:

def entropy(vector):
    # """ Fonction utilitaire pour calculer l'entropie sur un vecteur. """
    # H(S) = - sum(p_i * log2(p_i))  
    # On compte le nombre d'occurrences de chaque classe (Oui, Non) en utilisant la fonction 'unique'
    # avec return_counts=True pour retourner les occurrences
    _, counts = np.unique(vector, return_counts=True)
    # On calcule la probabilité de chaque classe (occurrence / nombre total d'éléments)
    probs = counts / len(vector)
    # On ajoute 10**-9 (très petit nombre) pour éviter log2(0) qui return inf
    return -np.sum(probs * np.log2(probs + 10**-9))


def split_data(data, index):
    # Divise le Dataset en sous-ensembles selon les valeurs uniques d'une colonne.
    # On récupère toutes les valeurs uniques de la colonne index
    values = np.unique(data[:, index])
    # On utilise un dictionnaire pour stocker chaque groupe
    return {v: data[data[:, index] == v] for v in values}

def gain(data, index):
    # Calcule le gain d'information.
    # Différence entre l'entropie parente et l'entropie moyenne des sous-ensembles.
    # premièrement, on calcule l'entropie totale
    total_entropy = entropy(data[:, -1])
    # on divise en sous-ensembles pour être capable de calculer la moyenne des entropies
    splits = split_data(data, index)
    
    weighted_entropy = sum((len(subset) / len(data)) * entropy(subset[:, -1]) 
                           for subset in splits.values())
    
    return total_entropy - weighted_entropy

def gain_ratio(data, index):
    # C4.5 utilise le Gain Ratio pour normaliser le Gain d'Information.
    # Formule : Gain Ratio = Gain Info / Split Info
    # avec split info = - sum(D_i * log2(D_i)) où D_i est la proportion de chaque sous-ensemble
    gain_info = gain(data, index)
    split_info = entropy(data[:, index])
    
    return gain_info / split_info if split_info != 0 else 0

def gini_impurity(data):
    # Mesure la probabilité de mal classer un élément.
    labels = data[:, -1]
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / len(labels)
    return 1 - np.sum(probs**2)

def gini_index(data, index):
    # Calcule l'Indice de Gini pondéré après séparation.
    # L'objectif de l'algorithme CART est de minimiser cette valeur.
    splits = split_data(data, index)
    
    return sum((len(subset) / len(data)) * gini_impurity(subset) 
               for subset in splits.values())

def majority_class(data):
    # Retourne la classe la plus fréquente (OUI ou NON).
    # Utile quand on s'arrête sur une feuille qui n'est pas pure à 100%.
    values, counts = np.unique(data[:, -1], return_counts=True)
    # on prend l'index de la valeur maximale avec argmax
    return values[np.argmax(counts)]

# 1.1. CODE SCRATCH (ID3)

In [2]:
import numpy as np

def id3(data, feature_indices, feature_names):
    # '''
    # on utilise feature_indices car il change à chaque fois dans la récursion (en enlevant l'index du meilleur gain)
    # la fonction sélectionne les autres features après avoir choisi la meilleure avec le gain,
    # et cela se répète jusqu'à ce qu'il n'y ait plus de features
    # '''
    # On extrait la dernière colonne qui contient les labels (OUI, NON)
    labels = data[:, -1]
    
    # Cas 1 : Si tous les exemples restants ont la même classe, on retourne cette classe (feuille pure)
    if np.all(labels == labels[0]):
        return labels[0]
    
    # Cas 2 : S'il n'y a plus de caractéristiques à tester, on retourne la classe majoritaire
    if len(feature_indices) == 0:
        return majority_class(data)
    
    # On calcule le gain d'information pour chaque caractéristique disponible
    gains = [gain(data, i) for i in feature_indices]
    
    # On sélectionne l'indice de la caractéristique avec le plus grand gain
    best_index_position = np.argmax(gains)
    # après, nous cherchons l'indice de ce best_index_position dans nos features pour obtenir la position
    best_feature_index = feature_indices[best_index_position]
    # après, nous sommes maintenant capables de retourner le nom de cette feature en nous basant sur son indice
    best_feature_name = feature_names[best_feature_index]
    
    # Initialisation du noeud courant avec le nom de la meilleure caractéristique
    tree = {best_feature_name: {}}
    
    # On sépare les données en fonction de cette meilleure caractéristique
    splits = split_data(data, best_feature_index)
    
    # Pour chaque sous-ensemble créé, on relance l'algorithme (récursivité)
    for value, subset in splits.items():
        # On retire la caractéristique qu'on vient d'utiliser pour ne pas la retester plus bas
        new_indices = [i for i in feature_indices if i != best_feature_index]
        # Appel récursif pour construire la branche
        tree[best_feature_name][value] = id3(subset, new_indices, feature_names)
    
    return tree


# 1.2.CODE SCRATCH (C4.5)

In [3]:
def c45(data, feature_indices, feature_names):
    # """ Algorithme C4.5 : Similaire à ID3 mais utilise le Gain Ratio au lieu de l'Information Gain. """
    labels = data[:, -1]
    
    if np.all(labels == labels[0]):
        return labels[0]
    
    if len(feature_indices) == 0:
        return majority_class(data)
    
    # On calcule le Gain Ratio plutôt du gain d'information
    gains = [gain_ratio(data, i) for i in feature_indices]
    
    # On prend la caractéristique qui MAXIMISE le Gain Ratio
    best_index_position = np.argmax(gains)
    best_feature_index = feature_indices[best_index_position]
    best_feature_name = feature_names[best_feature_index]
    
    tree = {best_feature_name: {}}
    splits = split_data(data, best_feature_index)
    
    for value, subset in splits.items():
        new_indices = [i for i in feature_indices if i != best_feature_index]
        tree[best_feature_name][value] = c45(subset, new_indices, feature_names)
    
    return tree

# 1.3.CODE SCRATCH (CART)

In [4]:
def cart(data, feature_indices, feature_names):
    # """ Algorithme CART en utilisant l'Indice de Gini. """
    labels = data[:, -1]
    
    if np.all(labels == labels[0]):
        return labels[0]
    
    if len(feature_indices) == 0:
        return majority_class(data)
    
    # On calcule l'Indice de Gini pour toutes les features
    ginis = [gini_index(data, i) for i in feature_indices]
    
    # CART cherche à MINIMISER le Gini Index (on utilise argmin plutôt que l'argmax)
    best_index_position = np.argmin(ginis) 
    best_feature_index = feature_indices[best_index_position]
    best_feature_name = feature_names[best_feature_index]
    
    tree = {best_feature_name: {}}
    splits = split_data(data, best_feature_index)
    
    for value, subset in splits.items():
        new_indices = [i for i in feature_indices if i != best_feature_index]
        tree[best_feature_name][value] = cart(subset, new_indices, feature_names)
    
    return tree

## a. DATASET

In [5]:
import numpy as np

data = np.array([
    ["Jeune", "Élevé", "Oui", "OUI"],
    ["Jeune", "Élevé", "Non", "OUI"],
    ["Moyen", "Élevé", "Non", "OUI"],
    ["Âgé", "Faible", "Non", "NON"],
    ["Âgé", "Faible", "Oui", "NON"],
    ["Âgé", "Faible", "Oui", "NON"],
    ["Moyen", "Faible", "Oui", "OUI"],
    ["Jeune", "Faible", "Non", "NON"]
])

features = ["Âge", "Revenu", "Étudiant"]



## b. Exécution du code scratch

In [6]:
# Generate initial indices [0, 1, 2] based on the number of features
initial_indices = list(range(len(features)))

# Run ID3
tree_Id3 = id3(data, initial_indices, features)

# Run C4.5
tree_c45 = c45(data, initial_indices, features)


# Run CART
tree_cart = cart(data, initial_indices, features)


UnboundLocalError: cannot access local variable 'gain' where it is not associated with a value

## c. Code sklearn

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import OrdinalEncoder

# Séparation des features (X) et de la variable cible (y)
X = data[:, :-1]
y = data[:, -1]

# Contrairement à notre code scratch, Sklearn ne peut pas  traiter des chaînes de caractères (strings).
# OrdinalEncoder transforme ["Jeune", "Moyen", "Âgé"] en valeurs numériques [0, 1, 2].
enc = OrdinalEncoder()
X_encoded = enc.fit_transform(X)

# Création du modèle. On utilise le critère 'entropy' pour qu'il imite la logique d'ID3 / C4.5
clf = DecisionTreeClassifier(criterion='entropy', random_state=42)

# Entraînement du modèle sklearn sur les données numérisées
sklearn_tree = clf.fit(X_encoded, y)


## d. Comparaison et graphiques (scratch/sklearn)

In [7]:
def print_tree(tree, indent=""):
    # Si 'tree' n'est plus un dictionnaire, c'est qu'on a atteint une feuille
    if not isinstance(tree, dict):
        print(indent + "|--- " + str(tree))
        return
    
    # Sinon, on parcourt les branches du noeud
    for i, (key, value) in enumerate(tree.items()):
        print(indent + "|---" + str(key))
        # On augmente l'indentation
        new_indent = indent + "│   "
        print_tree(value, new_indent)
        
   

print("Decision Tree ID3:")
print_tree(tree_Id3)
print("\n\n")

print("Arbre de Décision CART (Gini) :")
print_tree(tree_cart)
print("\n\n")


print("Arbre de Décision C4.5 :")
print_tree(tree_c45)
print("\n\n")


print("--- Sklearn Text Tree ---")
r = export_text(sklearn_tree, feature_names=features)
print(r)





Decision Tree ID3:
|---Âge
│   |---Jeune
│   │   |---Revenu
│   │   │   |---Faible
│   │   │   │   |--- NON
│   │   │   |---Élevé
│   │   │   │   |--- OUI
│   |---Moyen
│   │   |--- OUI
│   |---Âgé
│   │   |--- NON



Arbre de Décision CART (Gini) :


NameError: name 'tree_cart' is not defined